In [1]:
from pathlib import Path
import csv, json, shutil, hashlib, math, itertools, codecs

LAST_NAME = "Santarin"
STUDENT_ID = "TUPM-26-1865"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

print("Student:", LAST_NAME, "|", STUDENT_ID)
print("Seed:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Name Length:", NAME_LENGTH)
print("Workspace:", WORK_DIR.resolve())


def sha256_of(path, chunk_size=65536):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def backup_and_verify(src_path):
    """Copy a file into BACKUP_DIR and verify size + SHA-256 match."""
    src_path = Path(src_path)
    backup_path = BACKUP_DIR / src_path.name
    shutil.copy2(src_path, backup_path)
    verified = (
        src_path.stat().st_size == backup_path.stat().st_size
        and sha256_of(src_path) == sha256_of(backup_path)
    )
    return {
        "backup_path": str(backup_path),
        "size_bytes": backup_path.stat().st_size,
        "verified": verified,
    }


def to_nonneg_float(text):
    """Return a finite, non-negative float, or None if the value is invalid/missing."""
    try:
        value = float(str(text).strip())
    except (TypeError, ValueError):
        return None
    return value if math.isfinite(value) and value >= 0 else None


def require_file(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"Dataset not found: {path}. Place it in the data folder.")


def find_dataset(keyword):
    """Find a CSV, text, or log dataset by filename under DATA_DIR."""
    candidates = sorted(
        path for path in DATA_DIR.rglob("*")
        if path.is_file()
        and keyword.casefold() in path.name.casefold()
        and path.suffix.casefold() in {".csv", ".txt", ".log"}
    )
    if not candidates:
        raise FileNotFoundError(
            f"No dataset matching {keyword!r} found in {DATA_DIR.resolve()}. "
            "Place the weblog dataset in the data folder and rerun this cell."
        )
    return candidates[0]


def detect_encoding(path):
    """Detect UTF-8 (including BOM) or fall back to Windows-1252."""
    with open(path, "rb") as source:
        sample = source.read(65536)
    try:
        codecs.getincrementaldecoder("utf-8-sig")().decode(sample, final=False)
    except UnicodeDecodeError:
        return "cp1252"
    return "utf-8-sig"


def norm(value):
    """Normalize a column name for case- and punctuation-insensitive matching."""
    return "".join(character.lower() for character in str(value) if character.isalnum())

Student: Santarin | TUPM-26-1865
Seed: 5 | ID Sum: 28 | Name Length: 8
Workspace: C:\Users\Gab\Desktop\Lab6_Appendices


In [11]:
from pathlib import Path
import csv, json, shutil, hashlib, math, re, zipfile

# ---------- Configuration ----------
LAST_NAME = "Santarin"
STUDENT_ID = "TUPM-26-1865"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
for _folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    _folder.mkdir(exist_ok=True)

# ---------- Helpers ----------
SKIP_DIRS = {".venv", "venv", "output", "backup", ".git", "_pycache_", ".ipynb_checkpoints"}

def norm(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())

def _scan(pattern):
    return [p for p in WORK_DIR.rglob(pattern) if not any(part in SKIP_DIRS for part in p.parts)]

def find_dataset(keyword):
    key = norm(keyword)
    match = [p for p in _scan("*.csv") if key in norm(p.stem)]
    if not match:                                   # try unzipping Kaggle downloads
        for z in _scan("*.zip"):
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(DATA_DIR)
            except Exception:
                pass
        match = [p for p in _scan("*.csv") if key in norm(p.stem)]
    if not match:
        raise FileNotFoundError(
            f"No CSV with '{keyword}' in its name. Put the dataset inside the 'data' folder.\n"
            f"CSV files seen: {sorted(str(p) for p in _scan('*.csv')) or '(none)'}"
        )
    match.sort(key=lambda p: (p.parts[0] != "data", len(p.parts)))
    return match[0]

def detect_encoding(path):
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=enc, newline="") as f:
                for _ in f:
                    pass
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()

def backup_and_verify(src_path):
    src_path = Path(src_path)
    backup_path = BACKUP_DIR / src_path.name
    shutil.copy2(src_path, backup_path)
    verified = (src_path.stat().st_size == backup_path.stat().st_size
                and sha256_of(src_path) == sha256_of(backup_path))
    return {"backup_path": str(backup_path), "size_bytes": backup_path.stat().st_size, "verified": verified}

# ---------- Parameters ----------
dataset_path = find_dataset("weblog")
records_to_process = 1000 + ID_SUM              # 1028
target_class = (SEED_DIGIT % 4) + 2             # 3 -> 3xx
target_label = f"{target_class}xx"

filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_{target_label}_filtered.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"

records_read = 0
invalid_records = 0
selected_count = 0
status_counts = {}
url_counts = {}

# ---------- Read line by line (never loads the whole file) ----------
enc = detect_encoding(dataset_path)
with open(dataset_path, "r", encoding=enc, newline="") as source, \
     open(filtered_path, "w", encoding="utf-8", newline="") as output:

    writer = csv.writer(output)
    writer.writerow(["IP", "Time", "URL", "Status"])

    for row in csv.reader(source):
        row = [c.strip() for c in row]
        if len(row) < 4 or not any(row):
            continue

        # Skip the header line (its last column is not a number)
        if records_read == 0 and not row[-1].isdigit() and norm(row[0]) in ("ip", "ipaddress"):
            continue

        if records_read >= records_to_process:
            break
        records_read += 1

        ip, timestamp = row[0], row[1]
        status = row[-1]
        url = ",".join(row[2:-1]).strip()

        if not (ip and timestamp and url) or not (status.isdigit() and len(status) == 3):
            invalid_records += 1
            continue

        if int(status) // 100 == target_class:
            selected_count += 1
            writer.writerow([ip, timestamp, url, status])
            status_counts[status] = status_counts.get(status, 0) + 1
            url_counts[url] = url_counts.get(url, 0) + 1

# Five most requested URLs (stable sort keeps first-seen order on ties)
top_urls = sorted(url_counts.items(), key=lambda kv: kv[1], reverse=True)[:5]

# ---------- JSON summary + verified backup ----------
summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "records_processed": records_read,
    "invalid_records": invalid_records,
    "target_status_class": target_label,
    "records_in_target_class": selected_count,
    "count_per_status_code": dict(sorted(status_counts.items())),
    "top_5_urls": [{"url": u, "count": c} for u, c in top_urls],
    "filtered_csv": filtered_path.name,
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

backup_info = backup_and_verify(json_path)

# ---------- Assessment output ----------
print("Dataset used:", dataset_path)
print("Records processed:", records_read, f"(expected {records_to_process})")
print("Invalid/missing records skipped:", invalid_records)
print("Target status class:", target_label)
print("Records in target class:", selected_count)
print("Count per status code:", dict(sorted(status_counts.items())) or "No matching records")
print("Five most requested URLs:")
if top_urls:
    for rank, (u, c) in enumerate(top_urls, 1):
        print(f"  {rank}. {u} -> {c}")
else:
    print("  No matching records")
print("Filtered CSV file:", filtered_path.name, f"({filtered_path.stat().st_size} bytes)")
print("JSON summary file:", json_path.name, f"({json_path.stat().st_size} bytes)")
print("Backup verification:", "VERIFIED" if backup_info["verified"] else "CHECK", "->", backup_info["backup_path"])

Dataset used: Lab6_File_Handling_Additional_Exercises\weblog.csv
Records processed: 1028 (expected 1028)
Invalid/missing records skipped: 0
Target status class: 3xx
Records in target class: 165
Count per status code: {'302': 144, '304': 21}
Five most requested URLs:
  1. GET / HTTP/1.1 -> 46
  2. POST /process.php HTTP/1.1 -> 35
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 -> 34
  4. GET /home.php HTTP/1.1 -> 9
  5. GET /logout.php HTTP/1.1 -> 7
Filtered CSV file: Santarin_TUPM-26-1865_weblog_3xx_filtered.csv (12084 bytes)
JSON summary file: Santarin_TUPM-26-1865_weblog_summary.json (938 bytes)
Backup verification: VERIFIED -> backup\Santarin_TUPM-26-1865_weblog_summary.json


In [12]:
from pathlib import Path
import csv, json, shutil, hashlib, math, re, zipfile

# ---------- Configuration ----------
LAST_NAME = "Santarin"
STUDENT_ID = "TUPM-26-1865"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
for _folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    _folder.mkdir(exist_ok=True)

# ---------- Helpers ----------
SKIP_DIRS = {".venv", "venv", "output", "backup", ".git", "_pycache_", ".ipynb_checkpoints"}

def norm(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())

def _scan(pattern):
    return [p for p in WORK_DIR.rglob(pattern) if not any(part in SKIP_DIRS for part in p.parts)]

def find_dataset(keyword):
    key = norm(keyword)
    match = [p for p in _scan("*.csv") if key in norm(p.stem)]
    if not match:
        for z in _scan("*.zip"):
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(DATA_DIR)
            except Exception:
                pass
        match = [p for p in _scan("*.csv") if key in norm(p.stem)]
    if not match:
        raise FileNotFoundError(
            f"No CSV with '{keyword}' in its name. Put the dataset inside the 'data' folder.\n"
            f"CSV files seen: {sorted(str(p) for p in _scan('*.csv')) or '(none)'}"
        )
    match.sort(key=lambda p: (p.parts[0] != "data", len(p.parts)))
    return match[0]

def detect_encoding(path):
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=enc, newline="") as f:
                for _ in f:
                    pass
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"

def find_col(fieldnames, *aliases):
    mapping = {norm(f): f for f in fieldnames}
    for a in aliases:
        if norm(a) in mapping:
            return mapping[norm(a)]
    for a in aliases:
        for n, original in mapping.items():
            if norm(a) in n:
                return original
    raise KeyError(f"Column like {aliases} not found. Available columns: {list(fieldnames)}")

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()

def backup_and_verify(src_path):
    src_path = Path(src_path)
    backup_path = BACKUP_DIR / src_path.name
    shutil.copy2(src_path, backup_path)
    verified = (src_path.stat().st_size == backup_path.stat().st_size
                and sha256_of(src_path) == sha256_of(backup_path))
    return {"backup_path": str(backup_path), "size_bytes": backup_path.stat().st_size, "verified": verified}

def to_nonneg_float(text):
    try:
        value = float(str(text).strip().replace(",", ""))
    except (TypeError, ValueError):
        return None
    return value if math.isfinite(value) and value >= 0 else None

def to_int(text):
    try:
        return int(float(str(text).strip()))
    except (TypeError, ValueError):
        return None

def clean_row(row):
    return {k: ("" if v is None else v) for k, v in row.items() if k is not None}

# ---------- Parameters ----------
dataset_path = find_dataset("userbehavior")
target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"      # iOS
target_behavior = (SEED_DIGIT % 5) + 1                       # 1

exported_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_selected_users.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"

total_records = 0
os_match_count = 0
selected_count = 0
invalid_records = 0
battery_total = 0.0
data_total = 0.0
numeric_count = 0

# ---------- Read, filter and export ----------
enc = detect_encoding(dataset_path)
with open(dataset_path, "r", encoding=enc, newline="") as source, \
     open(exported_path, "w", encoding="utf-8", newline="") as output:

    reader = csv.DictReader(source)
    if not reader.fieldnames:
        raise ValueError("The dataset file is empty.")
    reader.fieldnames = [name.strip() for name in reader.fieldnames]
    fields = reader.fieldnames

    id_col = find_col(fields, "User ID")
    model_col = find_col(fields, "Device Model")
    os_col = find_col(fields, "Operating System")
    battery_col = find_col(fields, "Battery Drain (mAh/day)", "Battery Drain")
    data_col = find_col(fields, "Data Usage (MB/day)", "Data Usage")
    class_col = find_col(fields, "User Behavior Class", "Behavior Class")

    export_cols = [id_col, model_col, os_col, battery_col, data_col]
    writer = csv.writer(output)
    writer.writerow(["User ID", "Device Model", "Operating System",
                     "Battery Drain (mAh/day)", "Data Usage (MB/day)"])

    for raw in reader:
        row = clean_row(raw)
        total_records += 1

        if row.get(os_col, "").strip().lower() != target_os.lower():
            continue
        os_match_count += 1

        behavior = to_int(row.get(class_col, ""))
        if behavior is None:
            invalid_records += 1
            continue
        if behavior != target_behavior:
            continue

        selected_count += 1
        writer.writerow([row.get(c, "").strip() for c in export_cols])

        battery = to_nonneg_float(row.get(battery_col, ""))
        data_usage = to_nonneg_float(row.get(data_col, ""))
        if battery is None or data_usage is None:
            invalid_records += 1
            continue
        battery_total += battery
        data_total += data_usage
        numeric_count += 1

avg_battery = round(battery_total / numeric_count, 2) if numeric_count else None
avg_data = round(data_total / numeric_count, 2) if numeric_count else None

# ---------- Count verification (rows in exported CSV vs selected count) ----------
with open(exported_path, "r", encoding="utf-8", newline="") as f:
    exported_rows = max(sum(1 for _ in csv.reader(f)) - 1, 0)
count_verified = exported_rows == selected_count

# ---------- JSON summary + verified backup ----------
summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "operating_system": target_os,
    "target_behavior_class": target_behavior,
    "total_records": total_records,
    "records_matching_os": os_match_count,
    "records_in_target_class": selected_count,
    "average_battery_drain_mAh_day": avg_battery,
    "average_data_usage_MB_day": avg_data,
    "invalid_records": invalid_records,
    "exported_rows": exported_rows,
    "count_verified": count_verified,
    "exported_csv": exported_path.name,
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

backup_info = backup_and_verify(exported_path)

# ---------- Assessment output ----------
print("Dataset used:", dataset_path)
print("Operating system and behavior class:", target_os, "| Class", target_behavior)
print("Total records in dataset:", total_records)
print("Records matching operating system:", os_match_count)
print("Records in target behavior class:", selected_count)
print("Average battery drain (mAh/day):", avg_battery if avg_battery is not None else "N/A (no matching records)")
print("Average data usage (MB/day):", avg_data if avg_data is not None else "N/A (no matching records)")
print("Count verification:", "VERIFIED" if count_verified else "CHECK",
      f"({exported_rows} exported vs {selected_count} selected)")
print("Exported CSV file:", exported_path.name, f"({exported_path.stat().st_size} bytes)")
print("JSON summary file:", json_path.name, f"({json_path.stat().st_size} bytes)")
print("Backup file and size:", Path(backup_info["backup_path"]).name,
      f"({backup_info['size_bytes']} bytes)", "| Verified:", backup_info["verified"])

Dataset used: Lab6_File_Handling_Additional_Exercises\user_behavior_dataset.csv
Operating system and behavior class: iOS | Class 1
Total records in dataset: 700
Records matching operating system: 146
Records in target behavior class: 22
Average battery drain (mAh/day): 467.23
Average data usage (MB/day): 201.45
Count verification: VERIFIED (22 exported vs 22 selected)
Exported CSV file: Santarin_TUPM-26-1865_mobile_selected_users.csv (674 bytes)
JSON summary file: Santarin_TUPM-26-1865_mobile_summary.json (493 bytes)
Backup file and size: Santarin_TUPM-26-1865_mobile_selected_users.csv (674 bytes) | Verified: True


In [13]:
from pathlib import Path
import csv, json, shutil, hashlib, math, re, zipfile

# ---------- Configuration ----------
LAST_NAME = "Santarin"
STUDENT_ID = "TUPM-26-1865"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
for _folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    _folder.mkdir(exist_ok=True)

# ---------- Helpers ----------
SKIP_DIRS = {".venv", "venv", "output", "backup", ".git", "_pycache_", ".ipynb_checkpoints"}

def norm(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())

def _scan(pattern):
    return [p for p in WORK_DIR.rglob(pattern) if not any(part in SKIP_DIRS for part in p.parts)]

def find_dataset(keyword):
    key = norm(keyword)
    match = [p for p in _scan("*.csv") if key in norm(p.stem)]
    if not match:
        for z in _scan("*.zip"):
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(DATA_DIR)
            except Exception:
                pass
        match = [p for p in _scan("*.csv") if key in norm(p.stem)]
    if not match:
        raise FileNotFoundError(
            f"No CSV with '{keyword}' in its name. Put the dataset inside the 'data' folder.\n"
            f"CSV files seen: {sorted(str(p) for p in _scan('*.csv')) or '(none)'}"
        )
    match.sort(key=lambda p: (p.parts[0] != "data", len(p.parts)))
    return match[0]

def detect_encoding(path):
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=enc, newline="") as f:
                for _ in f:
                    pass
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"

def find_col(fieldnames, *aliases, required=True):
    mapping = {norm(f): f for f in fieldnames}
    for a in aliases:
        if norm(a) in mapping:
            return mapping[norm(a)]
    for a in aliases:
        for n, original in mapping.items():
            if norm(a) in n:
                return original
    if required:
        raise KeyError(f"Column like {aliases} not found. Available columns: {list(fieldnames)}")
    return None

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()

def backup_and_verify(src_path):
    src_path = Path(src_path)
    backup_path = BACKUP_DIR / src_path.name
    shutil.copy2(src_path, backup_path)
    verified = (src_path.stat().st_size == backup_path.stat().st_size
                and sha256_of(src_path) == sha256_of(backup_path))
    return {"backup_path": str(backup_path), "size_bytes": backup_path.stat().st_size, "verified": verified}

def to_nonneg_float(text):
    try:
        value = float(str(text).strip().replace(",", ""))
    except (TypeError, ValueError):
        return None
    return value if math.isfinite(value) and value >= 0 else None

def to_int(text):
    try:
        return int(float(str(text).strip()))
    except (TypeError, ValueError):
        return None

def clean_row(row):
    return {k: ("" if v is None else v) for k, v in row.items() if k is not None}

# ---------- Parameters ----------
dataset_path = find_dataset("cybersecurity")
target_year = 2015 + SEED_DIGIT                 # 2020

filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_{target_year}_filtered.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_top5_incidents.json"
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cybersecurity_report.txt"

total_records = 0
selected = []            # (original_order, row_dict, loss_or_None)
attack_counts = {}

# ---------- Read and select incidents in the target year ----------
enc = detect_encoding(dataset_path)
with open(dataset_path, "r", encoding=enc, newline="") as source:
    reader = csv.DictReader(source)
    if not reader.fieldnames:
        raise ValueError("The dataset file is empty.")
    reader.fieldnames = [name.strip() for name in reader.fieldnames]
    fieldnames = reader.fieldnames

    year_col = find_col(fieldnames, "Year")
    attack_col = find_col(fieldnames, "Attack Type")
    loss_col = find_col(fieldnames, "Financial Loss (in Million $)", "Financial Loss")
    country_col = find_col(fieldnames, "Country", required=False)
    industry_col = find_col(fieldnames, "Target Industry", required=False)

    for order, raw in enumerate(reader):
        row = clean_row(raw)
        total_records += 1

        if to_int(row.get(year_col, "")) != target_year:
            continue

        attack = row.get(attack_col, "").strip() or "Unknown"
        attack_counts[attack] = attack_counts.get(attack, 0) + 1
        selected.append((order, row, to_nonneg_float(row.get(loss_col, ""))))

incidents_in_year = len(selected)
valid_losses = [loss for _, _, loss in selected if loss is not None]
invalid_loss_count = incidents_in_year - len(valid_losses)

total_loss = round(sum(valid_losses), 2)
average_loss = round(total_loss / len(valid_losses), 2) if valid_losses else None

# ---------- Five costliest: descending loss, ties keep original dataset order ----------
valid_incidents = [item for item in selected if item[2] is not None]
valid_incidents.sort(key=lambda item: (-item[2], item[0]))
top_five = valid_incidents[:5]

with open(json_path, "w", encoding="utf-8") as f:
    json.dump([row for _, row, _ in top_five], f, indent=4)

backup_info = backup_and_verify(json_path)

# ---------- Filtered CSV (all incidents in the target year) ----------
with open(filtered_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    for _, row, _ in selected:
        writer.writerow(row)

# ---------- Text report ----------
sorted_counts = sorted(attack_counts.items(), key=lambda kv: kv[1], reverse=True)

def describe(row, loss):
    parts = [row.get(attack_col, "")]
    if country_col:
        parts.append(row.get(country_col, ""))
    if industry_col:
        parts.append(row.get(industry_col, ""))
    parts.append(f"${loss} M")
    return " | ".join(parts)

with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-" * 45 + "\n")
    report.write(f"Last Name: {LAST_NAME}\n")
    report.write(f"Student ID: {STUDENT_ID}\n")
    report.write(f"Seed Digit: {SEED_DIGIT}\n")
    report.write(f"Target Year: {target_year}\n")
    report.write(f"Total Records in Dataset: {total_records}\n")
    report.write(f"Incidents in Target Year: {incidents_in_year}\n")
    report.write(f"Invalid Financial Values: {invalid_loss_count}\n\n")

    report.write("INCIDENT COUNT PER ATTACK TYPE\n")
    if sorted_counts:
        for attack, count in sorted_counts:
            report.write(f"  {attack}: {count}\n")
    else:
        report.write("  No incidents found for the target year\n")

    report.write("\nFINANCIAL LOSS (USD millions)\n")
    report.write(f"  Total: {total_loss}\n")
    report.write(f"  Average: {average_loss if average_loss is not None else 'N/A'}\n\n")

    report.write("FIVE COSTLIEST INCIDENTS\n")
    if top_five:
        for rank, (_, row, loss) in enumerate(top_five, 1):
            report.write(f"  {rank}. {describe(row, loss)}\n")
    else:
        report.write("  No valid incidents available\n")

    report.write("\nOUTPUT FILES\n")
    report.write(f"  {filtered_path.name} ({filtered_path.stat().st_size} bytes)\n")
    report.write(f"  {json_path.name} ({json_path.stat().st_size} bytes)\n")
    report.write(f"  Backup: {Path(backup_info['backup_path']).name} "
                 f"({backup_info['size_bytes']} bytes) - "
                 f"{'VERIFIED' if backup_info['verified'] else 'CHECK'}\n")

# ---------- Assessment output ----------
print("Dataset used:", dataset_path)
print("Target year:", target_year)
print("Total records in dataset:", total_records)
print("Incidents in target year:", incidents_in_year, f"(invalid loss values: {invalid_loss_count})")
print("Count per attack type:", dict(sorted_counts) or "No incidents")
print("Total financial loss (USD millions):", total_loss)
print("Average financial loss (USD millions):", average_loss if average_loss is not None else "N/A")
print("Five costliest incidents:")
if top_five:
    for rank, (_, row, loss) in enumerate(top_five, 1):
        print(f"  {rank}. {describe(row, loss)}")
else:
    print("  None")
print("Filtered CSV file:", filtered_path.name, f"({filtered_path.stat().st_size} bytes)")
print("JSON summary file:", json_path.name, f"({json_path.stat().st_size} bytes)")
print("Backup verification:", "VERIFIED" if backup_info["verified"] else "CHECK")
print("Report file:", report_path.name, f"({report_path.stat().st_size} bytes)")

Dataset used: Lab6_File_Handling_Additional_Exercises\Global_Cybersecurity_Threats_2015-2024.csv
Target year: 2020
Total records in dataset: 3000
Incidents in target year: 315 (invalid loss values: 0)
Count per attack type: {'Phishing': 61, 'Ransomware': 59, 'DDoS': 56, 'SQL Injection': 55, 'Man-in-the-Middle': 47, 'Malware': 37}
Total financial loss (USD millions): 15767.95
Average financial loss (USD millions): 50.06
Five costliest incidents:
  1. Ransomware | Brazil | Banking | $99.9 M
  2. DDoS | Japan | Education | $99.83 M
  3. DDoS | USA | Banking | $99.81 M
  4. SQL Injection | India | Government | $99.72 M
  5. Phishing | USA | Retail | $99.64 M
Filtered CSV file: Santarin_TUPM-26-1865_incidents_2020_filtered.csv (26718 bytes)
JSON summary file: Santarin_TUPM-26-1865_top5_incidents.json (2223 bytes)
Backup verification: VERIFIED
Report file: Santarin_TUPM-26-1865_cybersecurity_report.txt (927 bytes)
